# ML-09 - Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sinagdevv/flyrank-ml-internship-sinagdev/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This notebook audits the Lane 2 model-selection claim rather than optimizing the score. It uses the authorized processed starter data and the paper available at `docs/flyrank-seo-research-march-2026.pdf`.

## 1. Two paper findings + my methodology questions

**Finding 1 (paper p. 6, "The Anatomy of Growing Content"):** The paper reports that growing pages average about 3.2K words and 184 days old versus about 2.3K words and 230 days for declining pages, using a direct portfolio comparison. The evidence is an observational up-versus-down group table, not a randomized refresh test. **Question:** Are the direction groups comparable in exposure, content type, and measurement coverage? For Lane 2, the same question applies because the starter proxy is also a current trend direction and can reflect selection or lifecycle differences rather than a refresh effect.

**Finding 2 (paper p. 8, "Click Capture by Position Tier"):** Portfolio-weighted CTR is reported as 0.423% in the top three, 0.339% on page one, and 0.050% deep, an 88% drop from top three to deep. The evidence is an aggregate weighted comparison by position tier, explicitly not a generic Google benchmark. **Question:** Are impressions and page mix sufficiently comparable across tiers, and how stable is the result across clients and time? For Lane 2, position and CTR are legitimate decision-time features, but the aggregate pattern does not by itself validate a future decline model.

In [1]:
import pandas as pd
paper_findings = pd.DataFrame([
    {"source": "Paper p. 6", "finding": "Growing pages are longer and younger than declining pages", "evidence": "Direct up/down portfolio comparison; observational group averages", "methodology_question": "Could exposure, content type, or coverage differ by direction?"},
    {"source": "Paper p. 8", "finding": "Weighted CTR falls from top-three positions to deep results", "evidence": "Aggregate weighted CTR by position tier", "methodology_question": "How stable is the tier contrast across clients and time?"},
])
display(paper_findings)
print("Paper source verified locally: docs/flyrank-seo-research-march-2026.pdf, pages 6 and 8.")

Paper source verified locally: docs/flyrank-seo-research-march-2026.pdf, pages 6 and 8.


,source,finding,evidence,methodology_question
0,Paper p. 6,Growing pages are longer and younger than declining pages,Direct up/down portfolio comparison; observational group averages,"Could exposure, content type, or coverage differ by direction?"
1,Paper p. 8,Weighted CTR falls from top-three positions to deep results,Aggregate weighted CTR by position tier,How stable is the tier contrast across clients and time?


## 2. My model under an honest split (before/after)

ML-08 selected Random Forest after comparing three models on the same outer client holdout, then reported that holdout. That is model-selection bias: the reported 0.760 is not an untouched final estimate. I use a new outer client-grouped holdout (seed 2026) and an inner grouped split (seed 43) for model selection. The baseline and model are evaluated on identical outer rows; Precision@50 remains primary. This is still a single snapshot and not prospective forecasting.

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

repo_root = Path.cwd()
if not (repo_root / "data/processed/refresh_feature_vector.csv").exists(): repo_root = (repo_root / "../..").resolve()
frame = pd.read_csv(repo_root / "data/processed/refresh_feature_vector.csv")
baseline = pd.read_csv(repo_root / "data/processed/baseline_refresh_queue.csv")
features = ["impressions_90d", "days_since_last_update", "avg_position", "ctr", "engagement_rate"]
y = frame["is_declining_label"].astype(int)

def grouped_split(data, seed):
    clients = data["client_id"].drop_duplicates().to_numpy(); rng = np.random.default_rng(seed)
    held = set(rng.permutation(clients)[:max(1, int(round(len(clients) * 0.2)))])
    mask = data["client_id"].isin(held).to_numpy()
    return np.flatnonzero(~mask), np.flatnonzero(mask)

def make_model(name):
    estimators = {"logistic_regression": LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42), "decision_tree": DecisionTreeClassifier(class_weight="balanced", max_depth=5, min_samples_leaf=50, random_state=42), "random_forest": RandomForestClassifier(class_weight="balanced_subsample", max_depth=10, min_samples_leaf=25, n_estimators=200, n_jobs=-1, random_state=42)}
    pre = ColumnTransformer([('numeric', Pipeline([('impute', SimpleImputer(strategy='median')), ('scale', StandardScaler())]), features)])
    return Pipeline([('preprocess', pre), ('model', estimators[name])])

def precision_at_50(labels, scores):
    order = np.argsort(-np.asarray(scores))[:50]; return float(np.asarray(labels)[order].mean())

def evaluate(model, train_idx, test_idx):
    model.fit(frame.iloc[train_idx][features], y.iloc[train_idx]); return model.predict_proba(frame.iloc[test_idx][features])[:, 1]

# Original ML-08 holdout: used to select RF and therefore optimistic as a final estimate.
original_train, original_test = grouped_split(frame, 42)
original_rf = evaluate(make_model("random_forest"), original_train, original_test)
lookup = baseline.set_index("content_id")["baseline_refresh_score"]
original_baseline = frame.iloc[original_test]["content_id"].map(lookup).fillna(0).to_numpy()

# New nested grouped design: inner split selects the model; outer split is evaluated once.
outer_train, outer_test = grouped_split(frame, 2026)
inner_rel_train, inner_rel_test = grouped_split(frame.iloc[outer_train].reset_index(drop=True), 43)
inner_train, inner_test = outer_train[inner_rel_train], outer_train[inner_rel_test]
inner_scores = {}
for name in ["logistic_regression", "decision_tree", "random_forest"]:
    inner_scores[name] = precision_at_50(y.iloc[inner_test], evaluate(make_model(name), inner_train, inner_test))
selected_model = max(inner_scores, key=inner_scores.get)
outer_model = evaluate(make_model(selected_model), outer_train, outer_test)
outer_baseline = frame.iloc[outer_test]["content_id"].map(lookup).fillna(0).to_numpy()
comparison = pd.DataFrame([
    {"evaluation": "ML-08 selected holdout", "model": "random_forest", "rows": len(original_test), "clients": len(set(frame.iloc[original_test].client_id)), "baseline_precision_at_50": precision_at_50(y.iloc[original_test], original_baseline), "model_precision_at_50": precision_at_50(y.iloc[original_test], original_rf)},
    {"evaluation": "Nested outer holdout", "model": selected_model, "rows": len(outer_test), "clients": len(set(frame.iloc[outer_test].client_id)), "baseline_precision_at_50": precision_at_50(y.iloc[outer_test], outer_baseline), "model_precision_at_50": precision_at_50(y.iloc[outer_test], outer_model)},
])
comparison["model_minus_baseline"] = comparison.model_precision_at_50 - comparison.baseline_precision_at_50
print(f"Inner selection scores: {inner_scores}; selected={selected_model}")
print(f"Outer split sizes: train={len(outer_train):,}, test={len(outer_test):,}; client overlap={len(set(frame.iloc[outer_train].client_id) & set(frame.iloc[outer_test].client_id))}")
display(comparison.round(3))

Inner selection scores: {'logistic_regression': 0.42, 'decision_tree': 0.82, 'random_forest': 0.74}; selected=decision_tree
Outer split sizes: train=21,179, test=8,821; client overlap=0


,evaluation,model,rows,clients,baseline_precision_at_50,model_precision_at_50,model_minus_baseline
0,ML-08 selected holdout,random_forest,2325,6,0.24,0.76,0.52
1,Nested outer holdout,decision_tree,8821,6,0.28,0.54,0.26


## 3. Leakage audit

The five model inputs are decision-time aggregates. IDs are grouping context only. `trend_direction` and `trend_pct` are target-derived; the 90-day/30-day snapshot also cannot establish a future label window. Imputation and scaling are inside the fitted training pipeline.

In [3]:
audit = pd.DataFrame([
    {"field": "impressions_90d", "decision_time": "yes", "target_derived": "no", "future": "no", "identifier": "no", "status": "ALLOWED", "why": "Exposure available in the snapshot before review."},
    {"field": "days_since_last_update / avg_position / ctr / engagement_rate", "decision_time": "yes", "target_derived": "no", "future": "no", "identifier": "no", "status": "ALLOWED", "why": "Freshness and performance context; preprocessing fits train only."},
    {"field": "trend_direction / trend_pct", "decision_time": "current outcome", "target_derived": "yes", "future": "not future, but label source", "identifier": "no", "status": "EXCLUDED", "why": "Defines the observed decline proxy."},
    {"field": "content_id / client_id", "decision_time": "context", "target_derived": "no", "future": "no", "identifier": "yes", "status": "EXCLUDED", "why": "Join, output, and grouped split only."},
])
display(audit)
assert set(features).isdisjoint({"content_id", "client_id", "trend_direction", "trend_pct", "is_declining_label"})
assert set(frame.iloc[outer_train].client_id).isdisjoint(set(frame.iloc[outer_test].client_id))
print("Leakage checks: PASS - no target-derived features, no identifiers as features, train-only preprocessing, zero outer client overlap.")

Leakage checks: PASS - no target-derived features, no identifiers as features, train-only preprocessing, zero outer client overlap.


,field,decision_time,target_derived,future,identifier,status,why
0,impressions_90d,yes,no,no,no,ALLOWED,Exposure available in the snapshot before review.
1,days_since_last_update / avg_position / ctr / engagement_rate,yes,no,no,no,ALLOWED,Freshness and performance context; preprocessing fits train only.
2,trend_direction / trend_pct,current outcome,yes,"not future, but label source",no,EXCLUDED,Defines the observed decline proxy.
3,content_id / client_id,context,no,no,yes,EXCLUDED,"Join, output, and grouped split only."


## 4. Claim rewrite

The original ML-08 statement that Random Forest "predicts which pages will decline" is too strong because the label is a current snapshot proxy and the same holdout selected the model. The replacement below is limited to observed ranking performance.

In [4]:
claims = pd.DataFrame([
    {"before": "Random Forest predicts which pages will decline.", "classification": "UNSUPPORTED", "after": "On the ML-08 client holdout, Random Forest ranked pages with the observed current decline proxy above the rule baseline, but that holdout was used for model selection and is optimistic."},
    {"before": "The model improves refresh outcomes.", "classification": "UNSUPPORTED", "after": "The score is decision-support for human review; this snapshot does not measure refresh outcomes or causal effects."},
    {"before": "The nested audit proves the model generalizes.", "classification": "UNSUPPORTED", "after": "In the new outer client-grouped split, the selected model and baseline produced the measured Precision@50 values shown above; this is directional evidence from one snapshot, not proof of prospective generalization."},
])
display(claims)
outer_scores = outer_model
outer_error_frame = frame.iloc[outer_test][["content_id", "client_id", "impressions_90d", "days_since_last_update", "avg_position", "ctr"]].copy()
outer_error_frame["score"] = outer_scores
outer_error_frame["observed_decline"] = y.iloc[outer_test].to_numpy()
outer_error_frame["predicted_decline"] = (outer_scores >= 0.5).astype(int)
print("False-positive examples from the outer holdout:")
display(outer_error_frame[(outer_error_frame.observed_decline == 0) & (outer_error_frame.predicted_decline == 1)].head(3))
print("False-negative examples from the outer holdout:")
display(outer_error_frame[(outer_error_frame.observed_decline == 1) & (outer_error_frame.predicted_decline == 0)].head(3))
print("Highly ranked weak recommendations to review:")
display(outer_error_frame.sort_values("score", ascending=False).head(3))
print("Safe claim: the nested result is measured and directional, not causal or future forecasting.")

False-positive examples from the outer holdout:
False-negative examples from the outer holdout:
Highly ranked weak recommendations to review:
Safe claim: the nested result is measured and directional, not causal or future forecasting.


,before,classification,after
0,Random Forest predicts which pages will decline.,UNSUPPORTED,"On the ML-08 client holdout, Random Forest ranked pages with the observed current decline proxy above the rule baseline, but that holdout was used for model selection and is optimistic."
1,The model improves refresh outcomes.,UNSUPPORTED,The score is decision-support for human review; this snapshot does not measure refresh outcomes or causal effects.
2,The nested audit proves the model generalizes.,UNSUPPORTED,"In the new outer client-grouped split, the selected model and baseline produced the measured Precision@50 values shown above; this is directional evidence from one snapshot, not proof of prospective generalization."


,content_id,client_id,impressions_90d,days_since_last_update,avg_position,ctr,score,observed_decline,predicted_decline
7,content_a63219c6e95a,client_19581e27de,1724,22,21.2,0.06,0.715203,0,1
31,content_24ee79621dbf,client_19581e27de,1959,22,42.0,0.10,0.596833,0,1
33,content_d87a116e2c79,client_19581e27de,298,22,6.8,0.00,0.715203,0,1


,content_id,client_id,impressions_90d,days_since_last_update,avg_position,ctr,score,observed_decline,predicted_decline
24,content_0e23e310d404,client_19581e27de,29541,104,4.1,0.27,0.342456,1,0
111,content_46eaff5b4ae8,client_19581e27de,3,22,16.3,0.00,0.136125,1,0
142,content_2818c49edbe2,client_19581e27de,4174,104,3.1,1.01,0.429396,1,0


,content_id,client_id,impressions_90d,days_since_last_update,avg_position,ctr,score,observed_decline,predicted_decline
29910,content_76b3dbc0536d,client_19581e27de,5467,104,14.8,0.07,0.715203,0,1
17,content_761a44afda12,client_19581e27de,9449,22,7.3,0.07,0.715203,1,1
136,content_388ad123d7bb,client_19581e27de,129,22,22.8,0.00,0.715203,0,1


## Self-check

- [x] Two paper findings have verified page locations, evidence summaries, and constructive questions.
- [x] Original model-selection holdout is distinguished from the new nested outer holdout.
- [x] Leakage audit covers target-derived fields, IDs, timing, and train-only preprocessing.
- [x] Real false positives, false negatives, and weak recommendations are shown.
- [x] Claims are rewritten as observed/measured/directional decision-support statements.
- [ ] Commit and submit to the internship portal (intentionally not performed in this task).